# 🚀 Real GPU Paths — Chart Fine-Tuning & Edge Inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mattral/production-vlm-engineering/blob/main/notebooks/colab/04_gpu_verification_colab.ipynb)

Part of **[production-vlm-engineering](https://github.com/Mattral/production-vlm-engineering)**.

## Why this notebook exists

`vlm_chart_finetune` and `vlm_edge_inference` both have two code paths: a CPU-only fallback
(pipeline mechanics on synthetic data, no real model download) and a **real** path that
downloads an actual checkpoint from Hugging Face Hub and runs genuine QLoRA fine-tuning /
ONNX export+quantization. The dev/CI environment this repo is otherwise verified in has no GPU
and no network access to `huggingface.co` (`403 host_not_allowed`), so the real path had never
actually been run — this notebook is what closed that gap.

**v2 update:** the first real run of this notebook (thank you!) surfaced two genuine bugs that
are now fixed in the repo, not worked around in the notebook:
- `AutoModelForVision2Seq` was removed outright in `transformers` v5.0 (Jan 2026) — the code now
  tries `AutoModelForImageTextToText` first and falls back for older installs.
- `torch.onnx.export`'s dynamic-shape hint for image height/width failed against
  `google/vit-base-patch16-224`'s fixed position embeddings — the export now correctly produces
  one fixed-shape model per benchmarked image size instead of assuming one dynamic model can
  serve all of them.
- `onnxscript` is now a declared dependency of the `onnx` extra (`torch>=2.9` needs it separately
  for the exporter it defaults to; it stopped bundling it).

None of that needs a manual workaround anymore — plain `pip install -e ".[ml,onnx,cli]"` is enough.

**Before running:** `Runtime → Change runtime type → T4 GPU` (or better). Without a GPU,
Section A will fail fast with a clear message — `bitsandbytes` 4-bit quantization requires CUDA.

**Expected runtime:** ~15-25 min total (mostly `torch`/`transformers` install + the
`Qwen/Qwen2-VL-2B-Instruct` download, a few GB). No runtime restart needed.

**What to do when it's done:** see the last section — copy the final summary cell's output
back into the conversation, or download and attach the two `results.json` files.

In [ ]:
# ── GPU check ──────────────────────────────────────────────────────
import subprocess
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run.")
    print("   (vlm_edge_inference's ONNX export can still run CPU-only; vlm_chart_finetune's")
    print("   real QLoRA path cannot — bitsandbytes 4-bit requires CUDA.)")
else:
    print(gpu_check.stdout)


In [ ]:
# ── Setup: clone + install the real ml/onnx/cli extras ──────────────
# Deliberately NOT --no-deps here (unlike the other colab notebooks): this
# notebook's entire point is exercising the real torch/transformers/peft/
# bitsandbytes/onnxruntime/optimum/onnxscript path, so those dependencies
# must actually be installed, not skipped. No runtime restart needed —
# both bugs the first run of this notebook hit are now fixed in the repo
# itself (see the changelog cell text above), not papered over here.
import subprocess, sys, time

t0 = time.time()
subprocess.run(["git", "clone", "--quiet", "--depth=1",
                 "https://github.com/Mattral/production-vlm-engineering.git",
                 "/content/pvlm"], check=True)
%cd /content/pvlm

install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[ml,onnx,cli]", "--quiet"],
    capture_output=True, text=True,
)
print(install.stdout[-2000:])
if install.returncode != 0:
    print("STDERR:", install.stderr[-3000:])
print(f"✅ Setup complete in {time.time()-t0:.0f}s")


In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


---
## Section A — `vlm_chart_finetune`: real QLoRA fine-tuning

Checkpoint: **`Qwen/Qwen2-VL-2B-Instruct`** (pinned 2026-03-01, see
`configs/vlm_chart_finetune.yaml` — re-verify it still resolves before trusting numbers beyond
this run, checkpoints move). Runs 4-bit QLoRA (`peft` + `bitsandbytes`) on synthetic chart
images, reports zero-shot vs. fine-tuned faithfulness/grounding/numeric-accuracy scores plus
training time — the real version of the before/after table already in the repo's docs.

In [ ]:
# ── Run the actual installed CLI entry point ─────────────────────────
# Using the real `production-vlm` console script (not `python -m`) doubles
# as a live test of it on a genuinely different machine/OS than it was
# fixed and verified on this session.
import subprocess, time

t0 = time.time()
result_a = subprocess.run(
    ["production-vlm", "run-example", "vlm_chart_finetune"],
    capture_output=True, text=True, timeout=1800,
)
print(result_a.stdout[-4000:])
if result_a.returncode != 0:
    print("STDERR:", result_a.stderr[-4000:])
print(f"\n[exit code {result_a.returncode}, {time.time()-t0:.0f}s]")


In [ ]:
import json
from pathlib import Path

results_a = json.load(open("outputs/vlm_chart_finetune/results.json"))
print("ran_with_real_ml_stack:", results_a.get("ran_with_real_ml_stack"))
print(json.dumps(results_a, indent=2)[:3000])


---
## Section B — `vlm_edge_inference`: real ONNX export + quantization

Checkpoint: **`google/vit-base-patch16-224`** (deliberately small/tractable — swap for the
Qwen2-VL checkpoint above for a true apples-to-apples edge-deployment number once you have
GPU + network access routinely, per the config's own comment). Exports one fixed-shape ONNX
model per size in `benchmark.image_sizes`, applies dynamic INT8 quantization to each, and
reports the real latency/throughput/memory/accuracy before/after table.

In [ ]:
result_b = subprocess.run(
    ["production-vlm", "run-example", "vlm_edge_inference"],
    capture_output=True, text=True, timeout=900,
)
print(result_b.stdout[-4000:])
if result_b.returncode != 0:
    print("STDERR:", result_b.stderr[-4000:])
print(f"[exit code {result_b.returncode}]")


In [ ]:
results_b = json.load(open("outputs/vlm_edge_inference/results.json"))
print("ran_with_real_ml_stack:", results_b.get("ran_with_real_ml_stack"))
print(json.dumps(results_b, indent=2)[:3000])


---
## Copy this back

Run the cell below and paste its full output back into the conversation — or just download
and attach `outputs/vlm_chart_finetune/results.json` and `outputs/vlm_edge_inference/results.json`
directly (`Files` pane on the left → navigate to `pvlm/outputs/...` → right-click → Download).

In [ ]:
print("=" * 70)
print("SUMMARY — paste everything below back into the conversation")
print("=" * 70)
for name, r in [("vlm_chart_finetune", results_a), ("vlm_edge_inference", results_b)]:
    print(f"\n--- {name} ---")
    print("ran_with_real_ml_stack:", r.get("ran_with_real_ml_stack"))
    print("checkpoint:", r.get("checkpoint"))
    for k, v in r.items():
        if k not in ("ran_with_real_ml_stack", "checkpoint") and not isinstance(v, (list, dict)):
            print(f"{k}: {v}")


---
**Repo**: [github.com/Mattral/production-vlm-engineering](https://github.com/Mattral/production-vlm-engineering)  
**If Section A failed**: most likely causes are (1) no GPU runtime selected, (2) the pinned
checkpoint no longer resolves on Hugging Face Hub, or (3) a `bitsandbytes` CUDA version
mismatch — paste the full traceback back and it can be diagnosed properly rather than guessed
at, the same way both bugs the first run of this notebook hit were found: by looking at what
actually happened, not by assuming.